# FLAN-T5 SQuAD v2 question generation — fast Kaggle T4 version

Faster rewrite of the previous notebook. Pick `PRESET` in the config cell:

| Preset | Model | Data | Expected time on T4 |
|---|---|---|---|
| `fast` | flan-t5-small | full ~86k, 1 epoch | ~5-10 min |
| `balanced` | flan-t5-base | 30k random examples, 1 epoch | ~10-20 min |

Times are estimates: check the it/s in the progress bar during the first minute.

**What changed vs. the old notebook**
- fp32 weights + fp16 mixed precision. bf16 is used only on Ampere+ GPUs (T4 has no native bf16). Training stops early if the loss becomes NaN.
- Adafactor, lr 1e-3, warmup. The old run used the default 5e-5, which is too low for a short T5 run.
- `group_by_length`, `pad_to_multiple_of=8`, max input 384 and a bigger batch: much less padding waste.
- The subset is a *random* sample (the old one took the first N rows, i.e. only the first articles).
- No per-epoch checkpoints, and only the eval loss during training. BLEU/ROUGE come from one batched `generate()` pass at the end (the old `compute_metrics` received logits, not token IDs).

**Before running:** attach the dataset containing `squadv2_train.jsonl`, set Settings -> Accelerator -> GPU T4, then run all cells top to bottom.

In [40]:
import os
# Single GPU only (avoids DataParallel overhead when Kaggle shows "T4 x2")
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import glob, json, math, random, re, shutil, time
from pathlib import Path

import numpy as np
import torch

WORK = Path('/kaggle/working')
DATA_DIR = WORK / 'data' / 'training'
OUT_DIR = WORK / 'models' / 'question_generation' / 'flan-t5-squadv2'
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ============================ CONFIG (edit here) ============================
PRESET = 'fast'      # 'fast' or 'balanced' (see table above)
PRECISION = 'fp32'   # 'auto' | 'fp16' | 'bf16' | 'fp32'   (use 'fp32' if the loss turns NaN)

PRESETS = {
    'fast':     dict(model='google/flan-t5-small', batch=32, lr=1e-3, epochs=1, max_train=None),
    'balanced': dict(model='google/flan-t5-base',  batch=16, lr=1e-3, epochs=1, max_train=30000),
}
MAX_INPUT_LEN = 384      # SQuAD contexts are short; 512 mostly adds padding
MAX_TARGET_LEN = 64
EVAL_GEN_SAMPLES = 1000  # held-out samples used for the final BLEU/ROUGE
GEN_BATCH = 64
NUM_BEAMS = 4
SEED = 42
# ===========================================================================

cfg = PRESETS[PRESET]
MODEL_NAME, BATCH_SIZE, LR = cfg['model'], cfg['batch'], cfg['lr']
EPOCHS, MAX_TRAIN_SAMPLES = cfg['epochs'], cfg['max_train']

assert torch.cuda.is_available(), 'No GPU detected. Enable Settings -> Accelerator -> GPU T4.'
cap = torch.cuda.get_device_capability(0)
if PRECISION == 'auto':
    PRECISION = 'bf16' if cap[0] >= 8 else 'fp16'   # T4 (sm_75) has no native bf16
USE_BF16, USE_FP16 = PRECISION == 'bf16', PRECISION == 'fp16'
AMP_ENABLED = PRECISION in ('fp16', 'bf16')
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print('torch:', torch.__version__)
print('gpu:', torch.cuda.get_device_name(0), '| compute capability:', cap)
print('preset:', PRESET, '|', MODEL_NAME, '| batch', BATCH_SIZE, '| precision', PRECISION)

torch: 2.10.0+cu128
gpu: Tesla T4 | compute capability: (7, 5)
preset: fast | google/flan-t5-small | batch 32 | precision fp32


In [41]:
# Locate uploaded dataset files anywhere under /kaggle/input
def find_file(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return Path(hits[0]) if hits else None

train_src = find_file('squadv2_train.jsonl')
val_src = find_file('squadv2_validation.jsonl')
assert train_src is not None, 'squadv2_train.jsonl not found. Attach your Kaggle dataset first.'
print('train source:', train_src)

train_path = DATA_DIR / 'squadv2_train.jsonl'
val_path = DATA_DIR / 'squadv2_validation.jsonl'

if val_src is not None:
    shutil.copy(train_src, train_path)
    shutil.copy(val_src, val_path)
    print('validation source:', val_src)
else:
    # No validation file uploaded: carve a held-out split from train (no overlap).
    lines = train_src.read_text(encoding='utf-8').splitlines()
    rng = random.Random(SEED)
    val_idx = set(rng.sample(range(len(lines)), 2000))
    with open(train_path, 'w', encoding='utf-8') as ft, open(val_path, 'w', encoding='utf-8') as fv:
        for i, ln in enumerate(lines):
            (fv if i in val_idx else ft).write(ln + '\n')
    print(f'Carved 2000 held-out validation records; train now {len(lines) - 2000} examples.')

first = json.loads(open(train_path, encoding='utf-8').readline())
assert {'input_text', 'target_text'} <= set(first), f'Unexpected columns: {list(first)}'
print('train lines:', sum(1 for _ in open(train_path, encoding='utf-8')))
print('val lines:  ', sum(1 for _ in open(val_path, encoding='utf-8')))
print('example input :', first['input_text'][:200].replace('\n', ' '), '...')
print('example target:', first['target_text'])

train source: /kaggle/input/datasets/soykot124/squadv2/squadv2_train.jsonl
Carved 2000 held-out validation records; train now 84821 examples.
train lines: 84821
val lines:   2000
example input : generate question: context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Housto ...
example target: When did Beyonce start becoming popular?


In [ ]:
# Metrics deps
!pip install -q rouge-score nltk

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

set_seed(SEED)
dataset = load_dataset('json', data_files={'train': str(train_path), 'validation': str(val_path)})
dataset = dataset.filter(lambda ex: bool(ex['target_text']) and bool(ex['target_text'].strip()))

# Random subset (not the first N rows, which would only cover the first articles)
train_raw = dataset['train'].shuffle(seed=SEED)
if MAX_TRAIN_SAMPLES and MAX_TRAIN_SAMPLES < len(train_raw):
    train_raw = train_raw.select(range(MAX_TRAIN_SAMPLES))
eval_raw = dataset['validation'].shuffle(seed=SEED)
if len(eval_raw) > EVAL_GEN_SAMPLES:
    eval_raw = eval_raw.select(range(EVAL_GEN_SAMPLES))
print('train examples:', len(train_raw), '| held-out eval examples:', len(eval_raw))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# fp32 master weights; mixed precision is handled by the Trainer (autocast), NOT by casting the model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=MAX_TARGET_LEN, truncation=True)
    # padding is done dynamically by the collator; -100 is ignored by the loss
    inputs['labels'] = labels['input_ids']
    return inputs

train_tok = train_raw.map(tokenize, batched=True, num_proc=2,
                          remove_columns=train_raw.column_names, desc='Tokenizing train')
eval_tok = eval_raw.map(tokenize, batched=True,
                        remove_columns=eval_raw.column_names, desc='Tokenizing eval')

lens = [len(x) for x in train_tok.select(range(min(3000, len(train_tok))))['input_ids']]
print(f'input tokens: mean {np.mean(lens):.0f} | p95 {np.percentile(lens, 95):.0f} | max {max(lens)}'
      f' (truncated at {MAX_INPUT_LEN})')

In [ ]:
# BLEU / ROUGE helpers (used once, after training)
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu, corpus_bleu
from rouge_score import rouge_scorer

_TOK = re.compile(r"\w+|[^\w\s]")
def toks(s):
    return _TOK.findall(s.lower())

def compute_generation_metrics(preds, refs):
    smooth = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    em = b1 = b2 = b4 = r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        pt, rt = toks(p), toks(r)
        em += (pt == rt)
        if pt:
            b1 += sentence_bleu([rt], pt, weights=(1, 0, 0, 0), smoothing_function=smooth)
            b2 += sentence_bleu([rt], pt, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
            b4 += sentence_bleu([rt], pt, weights=(0.25,) * 4, smoothing_function=smooth)
        sc = scorer.score(r, p)
        r1 += sc['rouge1'].fmeasure; r2 += sc['rouge2'].fmeasure; rl += sc['rougeL'].fmeasure
    n = max(len(preds), 1)
    corpus_b4 = corpus_bleu([[toks(r)] for r in refs], [toks(p) for p in preds],
                            smoothing_function=smooth) * 100
    return {
        'exact_match': round(em / n, 4),
        'bleu1': round(b1 / n * 100, 2), 'bleu2': round(b2 / n * 100, 2),
        'bleu4_sentence': round(b4 / n * 100, 2),
        'bleu4_corpus': round(corpus_b4, 2),   # closer to how papers report BLEU-4
        'rouge1': round(r1 / n, 4), 'rouge2': round(r2 / n, 4), 'rougeL': round(rl / n, 4),
    }

# quick self-test
print(compute_generation_metrics(['what is the capital of france ?'], ['What is the capital of France?']))

In [ ]:
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          TrainerCallback)

class StopOnNaN(TrainerCallback):
    # fp16 can overflow on T5; stop immediately instead of wasting the run
    triggered = False
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs and not math.isfinite(logs['loss']):
            StopOnNaN.triggered = True
            control.should_training_stop = True
            print('\n!! Loss is NaN/inf -> stopping. Set PRECISION = "fp32" (or use the small model) and rerun.')

steps_per_epoch = math.ceil(len(train_tok) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(0.03 * total_steps))
print(f'{steps_per_epoch} steps/epoch | {total_steps} total | warmup {warmup_steps}')

OUT_DIR.mkdir(parents=True, exist_ok=True)
args = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    optim='adafactor',            # standard for T5, low memory, works well at lr 1e-3
    lr_scheduler_type='linear',
    warmup_steps=warmup_steps,
    weight_decay=0.0,
    fp16=USE_FP16,
    bf16=USE_BF16,
    group_by_length=True,         # batch similar lengths together -> less padding
    save_strategy='no',           # save once at the end
    predict_with_generate=False,  # only the eval LOSS during training; BLEU/ROUGE later
    logging_nan_inf_filter=False,   # show NaN losses instead of logging 0.0
    logging_steps=50,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to='none',
    seed=SEED,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)
try:
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=eval_tok,
                             processing_class=tokenizer, data_collator=collator,
                             callbacks=[StopOnNaN()])
except TypeError:
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=eval_tok,
                             tokenizer=tokenizer, data_collator=collator,
                             callbacks=[StopOnNaN()])

torch.cuda.empty_cache()

model.to('cuda')
batch = {k: v.to('cuda') for k, v in collator([train_tok[i] for i in range(8)]).items()}
with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=AMP_ENABLED):
    loss0 = model(**batch).loss.item()
print('preflight loss:', loss0)
assert math.isfinite(loss0), 'NaN/inf forward pass at this precision -> set PRECISION = "fp32"'

t_start = time.time()
train_result = trainer.train()
train_minutes = (time.time() - t_start) / 60
assert not StopOnNaN.triggered, 'Training hit NaN loss; nothing was saved. See message above.'

eval_metrics = trainer.evaluate()   # eval loss only (cheap)
if 'eval_loss' in eval_metrics:
    eval_metrics['eval_perplexity'] = round(math.exp(eval_metrics['eval_loss']), 3)

trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f'\nTraining done in {train_minutes:.1f} min | eval loss {eval_metrics.get("eval_loss"):.4f}'
      f' | perplexity {eval_metrics.get("eval_perplexity")}')

In [ ]:
# Final BLEU/ROUGE via ONE batched generate() pass on the held-out samples
model = trainer.model
model.eval()
torch.cuda.empty_cache()

inputs = eval_raw['input_text']
refs = eval_raw['target_text']
order = sorted(range(len(inputs)), key=lambda i: len(inputs[i]))   # similar lengths per batch = faster
preds = [None] * len(inputs)

t_gen = time.time()
with torch.no_grad(), torch.autocast('cuda', dtype=AMP_DTYPE, enabled=AMP_ENABLED):
    for s in range(0, len(order), GEN_BATCH):
        idx = order[s:s + GEN_BATCH]
        batch = tokenizer([inputs[i] for i in idx], max_length=MAX_INPUT_LEN, truncation=True,
                          padding=True, return_tensors='pt').to('cuda')
        out = model.generate(**batch, num_beams=NUM_BEAMS, max_new_tokens=MAX_TARGET_LEN,
                             early_stopping=True)
        for i, text in zip(idx, tokenizer.batch_decode(out, skip_special_tokens=True)):
            preds[i] = text.strip()
gen_seconds = time.time() - t_gen

gen_metrics = compute_generation_metrics(preds, refs)
print(f'Generated {len(preds)} questions in {gen_seconds:.0f}s ({len(preds) / gen_seconds:.1f} q/s)')
print(json.dumps(gen_metrics, indent=2))

print('\nSample outputs:')
for i in random.Random(SEED).sample(range(len(preds)), 5):
    print('  REF :', refs[i])
    print('  PRED:', preds[i])
    print()

run_record = {
    'preset': PRESET,
    'model_name': MODEL_NAME,
    'device': torch.cuda.get_device_name(0),
    'precision': PRECISION,
    'optimizer': 'adafactor',
    'learning_rate': LR,
    'train_examples': len(train_tok),
    'eval_examples': len(eval_raw),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'max_input_len': MAX_INPUT_LEN,
    'num_beams_eval': NUM_BEAMS,
    'train_minutes': round(train_minutes, 2),
    'seed': SEED,
    'train_metrics': train_result.metrics,
    'eval_loss_metrics': eval_metrics,
    'generation_metrics': gen_metrics,
}

(OUT_DIR / 'training_run.json').write_text(json.dumps(run_record, indent=2, default=str) + '\n', encoding='utf-8')

In [ ]:
# Zip checkpoint output
zip_path = shutil.make_archive(str(WORK / 'flan-t5-squadv2'), 'zip', root_dir=str(OUT_DIR))
print('Zipped checkpoint:', zip_path)
print()
print('Contents of the checkpoint folder:')
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print('  ', p.name, round(p.stat().st_size / 1e6, 1), 'MB')
print()
print('NEXT STEPS (local):')
print('  1. Download flan-t5-squadv2.zip from the Output panel.')
print('  2. Unzip into: models/question_generation/flan-t5-squadv2/')